In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

### Dataset Paths

In [ ]:
BASE_DIR = '/kaggle/input/datasets/ani3hhh/vr-dataset/data'

DATA_SPLITS = [
    (
        os.path.join(BASE_DIR, 'train', 'image'),
        os.path.join(BASE_DIR, 'train', 'annos')
    ),
    (
        os.path.join(BASE_DIR, 'validation', 'image'),
        os.path.join(BASE_DIR, 'json_for_validation')
    ),
    (
        os.path.join(BASE_DIR, 'test', 'image'),
        os.path.join(BASE_DIR, 'test', 'json_for_test')
    ),
]

TOP5_JSON      = os.path.join(BASE_DIR, 'top5.json')
LABEL_MAP_JSON = os.path.join(BASE_DIR, 'label_map_classification.json')
NUM_CLASSES    = 5

with open(TOP5_JSON) as f:
    top5 = json.load(f)

with open(LABEL_MAP_JSON) as f:
    label_map = {int(k): int(v) for k, v in json.load(f).items()}

top5_set = set(top5)

print('Top-5 categories:', top5)
print('Label map       :', label_map)

### Build Full Dataset from Raw Annotations (100% data)

In [ ]:
image_to_categories = {}

print('Scanning annotations across all splits...')

for IMAGE_DIR, ANN_DIR in DATA_SPLITS:

    if not os.path.isdir(ANN_DIR):
        print(f'  [WARN] Missing dir, skipping: {ANN_DIR}')
        continue

    for ann_file in sorted(os.listdir(ANN_DIR)):

        if not ann_file.endswith('.json'):
            continue

        with open(os.path.join(ANN_DIR, ann_file)) as f:
            ann_data = json.load(f)

        cats = set()
        for key in ann_data:
            if key.startswith('item'):
                cat = int(ann_data[key]['category_id'])
                if cat in top5_set:
                    cats.add(cat)

        if not cats:
            continue

        image_path = os.path.join(IMAGE_DIR, ann_file.replace('.json', '.jpg'))
        if not os.path.exists(image_path):
            continue

        image_to_categories[image_path] = list(cats)

print(f'Total valid images: {len(image_to_categories)}')

# Build multi-label vectors
all_images, all_labels = [], []
for img_path, cats in image_to_categories.items():
    vec = [0] * NUM_CLASSES
    for c in cats:
        vec[label_map[c]] = 1
    all_images.append(img_path)
    all_labels.append(vec)

print(f'Dataset entries  : {len(all_images)}')

### Train / Validation / Test Split  (70 / 15 / 15)

In [ ]:
train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(
    all_images, all_labels, test_size=0.30, random_state=SEED
)
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    temp_imgs, temp_labels, test_size=0.50, random_state=SEED
)

print(f'Train : {len(train_imgs)}')
print(f'Val   : {len(val_imgs)}')
print(f'Test  : {len(test_imgs)}')

### Dataset Class & Transforms

In [ ]:
class MultiLabelDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(self.labels[idx], dtype=torch.float32)


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

### DataLoaders

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = 2

train_dataset = MultiLabelDataset(train_imgs, train_labels, transform=train_transform)
val_dataset   = MultiLabelDataset(val_imgs,   val_labels,   transform=val_transform)
test_dataset  = MultiLabelDataset(test_imgs,  test_labels,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

### Class Weights (handle imbalance)

In [ ]:
label_array = np.array(train_labels)
pos_counts  = label_array.sum(axis=0).clip(min=1)
neg_counts  = (1 - label_array).sum(axis=0).clip(min=1)
pos_weight  = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(device)

print('Positive class weights:', pos_weight.cpu().numpy())

### Evaluation Helpers

In [ ]:
def evaluate(model, loader, tag=''):
    model.eval()
    all_probs, all_preds, all_gts = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            probs   = torch.sigmoid(outputs).cpu().numpy()
            all_probs.append(probs)
            all_preds.append((probs > 0.5).astype(int))
            all_gts.append(labels.numpy())

    all_probs = np.vstack(all_probs)
    all_preds = np.vstack(all_preds)
    all_gts   = np.vstack(all_gts)

    precision = precision_score(all_gts, all_preds, average=None, zero_division=0)
    recall    = recall_score(all_gts, all_preds, average=None, zero_division=0)
    f1_per    = f1_score(all_gts, all_preds, average=None, zero_division=0)
    f1_macro  = f1_score(all_gts, all_preds, average='macro', zero_division=0)
    f1_micro  = f1_score(all_gts, all_preds, average='micro', zero_division=0)
    auc       = roc_auc_score(all_gts, all_probs, average=None)

    print(f'\n=== {tag} ===')
    print(f"{'Class':<8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'AUC':>8}")
    for i in range(NUM_CLASSES):
        print(f'  [{i}]   {precision[i]:>10.4f} {recall[i]:>8.4f} {f1_per[i]:>8.4f} {auc[i]:>8.4f}')
    print(f'Macro F1 : {f1_macro:.4f}')
    print(f'Micro F1 : {f1_micro:.4f}')

    return all_probs, all_preds, all_gts


def plot_roc_curves(all_gts, all_probs, tag=''):
    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 4))
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(all_gts[:, i], all_probs[:, i])
        auc_val = roc_auc_score(all_gts[:, i], all_probs[:, i])
        axes[i].plot(fpr, tpr, label=f'AUC={auc_val:.3f}')
        axes[i].plot([0, 1], [0, 1], 'k--')
        axes[i].set_title(f'Class {i}')
        axes[i].set_xlabel('FPR')
        axes[i].set_ylabel('TPR')
        axes[i].legend()
    fig.suptitle(f'ROC Curves — {tag}')
    plt.tight_layout()
    plt.savefig(f"roc_{tag.lower().replace(' ', '_')}.png", dpi=120)
    plt.show()


def tune_thresholds(all_gts, all_probs):
    best_thresholds = []
    for i in range(NUM_CLASSES):
        best_f1, best_t = 0, 0.5
        for t in np.arange(0.1, 0.91, 0.05):
            f1 = f1_score(all_gts[:, i], (all_probs[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
    print('Best thresholds per class:', best_thresholds)
    return best_thresholds


def evaluate_tuned(all_gts, all_probs, thresholds, tag=''):
    tuned_preds = np.stack(
        [(all_probs[:, i] > thresholds[i]).astype(int) for i in range(NUM_CLASSES)], axis=1
    )
    print(f'\n=== {tag} — After Threshold Tuning ===')
    print(f"Precision : {precision_score(all_gts, tuned_preds, average=None, zero_division=0)}")
    print(f"Recall    : {recall_score(all_gts, tuned_preds, average=None, zero_division=0)}")
    print(f"Macro F1  : {f1_score(all_gts, tuned_preds, average='macro', zero_division=0):.4f}")
    print(f"Micro F1  : {f1_score(all_gts, tuned_preds, average='micro', zero_division=0):.4f}")

### Strategy 1 — Training from Scratch (ResNet-50)

In [ ]:
scratch_model = models.resnet50(weights=None)
scratch_model.fc = nn.Linear(scratch_model.fc.in_features, NUM_CLASSES)
scratch_model = scratch_model.to(device)

criterion_scratch = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_scratch = torch.optim.Adam(scratch_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_scratch = torch.optim.lr_scheduler.StepLR(optimizer_scratch, step_size=3, gamma=0.5)

print('Scratch model ready.')

In [ ]:
EPOCHS_SCRATCH = 10

for epoch in range(EPOCHS_SCRATCH):
    scratch_model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = criterion_scratch(scratch_model(images), labels)
        optimizer_scratch.zero_grad()
        loss.backward()
        optimizer_scratch.step()
        running_loss += loss.item()

    scheduler_scratch.step()
    print(f'[Scratch] Epoch {epoch+1}/{EPOCHS_SCRATCH}  '
          f'Loss: {running_loss:.4f}  '
          f'LR: {scheduler_scratch.get_last_lr()[0]:.2e}')

In [ ]:
scratch_probs, scratch_preds, scratch_gt = evaluate(scratch_model, val_loader, tag='Scratch — Validation')
plot_roc_curves(scratch_gt, scratch_probs, tag='Scratch Validation')

In [ ]:
scratch_thresholds = tune_thresholds(scratch_gt, scratch_probs)
evaluate_tuned(scratch_gt, scratch_probs, scratch_thresholds, tag='Scratch Validation')

In [ ]:
torch.save(scratch_model.state_dict(), 'resnet50_scratch.pth')
print('Saved: resnet50_scratch.pth')

### Strategy 2 — Transfer Learning (ResNet-50, ImageNet weights)

In [ ]:
transfer_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze entire backbone
for param in transfer_model.parameters():
    param.requires_grad = False

# Replace & unfreeze classifier head
transfer_model.fc = nn.Linear(transfer_model.fc.in_features, NUM_CLASSES)

transfer_model = transfer_model.to(device)

criterion_tl = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_tl = torch.optim.Adam(transfer_model.fc.parameters(), lr=1e-3)
scheduler_tl = torch.optim.lr_scheduler.StepLR(optimizer_tl, step_size=3, gamma=0.5)

print('Transfer model ready (backbone frozen).')

In [ ]:
# Phase 1 — train head only
EPOCHS_PHASE1 = 5

for epoch in range(EPOCHS_PHASE1):
    transfer_model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = criterion_tl(transfer_model(images), labels)
        optimizer_tl.zero_grad()
        loss.backward()
        optimizer_tl.step()
        running_loss += loss.item()

    scheduler_tl.step()
    print(f'[Transfer Phase-1] Epoch {epoch+1}/{EPOCHS_PHASE1}  Loss: {running_loss:.4f}')

In [ ]:
# Phase 2 — unfreeze layer3 + layer4 for fine-tuning
print('Unfreezing layer3 and layer4...')
for name, param in transfer_model.named_parameters():
    if any(k in name for k in ['layer3', 'layer4', 'fc']):
        param.requires_grad = True

optimizer_ft = torch.optim.Adam(
    filter(lambda p: p.requires_grad, transfer_model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler_ft = torch.optim.lr_scheduler.StepLR(optimizer_ft, step_size=3, gamma=0.5)

EPOCHS_PHASE2 = 5

for epoch in range(EPOCHS_PHASE2):
    transfer_model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = criterion_tl(transfer_model(images), labels)
        optimizer_ft.zero_grad()
        loss.backward()
        optimizer_ft.step()
        running_loss += loss.item()

    scheduler_ft.step()
    print(f'[Transfer Phase-2] Epoch {epoch+1}/{EPOCHS_PHASE2}  Loss: {running_loss:.4f}')

In [ ]:
transfer_probs, transfer_preds, transfer_gt = evaluate(transfer_model, val_loader, tag='Transfer — Validation')
plot_roc_curves(transfer_gt, transfer_probs, tag='Transfer Validation')

In [ ]:
transfer_thresholds = tune_thresholds(transfer_gt, transfer_probs)
evaluate_tuned(transfer_gt, transfer_probs, transfer_thresholds, tag='Transfer Validation')

In [ ]:
torch.save(transfer_model.state_dict(), 'resnet50_transfer.pth')
print('Saved: resnet50_transfer.pth')

### Final Test Set Evaluation

In [ ]:
# Scratch model on test set
s_test_probs, _, s_test_gt = evaluate(scratch_model, test_loader, tag='Scratch — Test')
plot_roc_curves(s_test_gt, s_test_probs, tag='Scratch Test')
evaluate_tuned(s_test_gt, s_test_probs, scratch_thresholds, tag='Scratch Test')

In [ ]:
# Transfer model on test set
t_test_probs, _, t_test_gt = evaluate(transfer_model, test_loader, tag='Transfer — Test')
plot_roc_curves(t_test_gt, t_test_probs, tag='Transfer Test')
evaluate_tuned(t_test_gt, t_test_probs, transfer_thresholds, tag='Transfer Test')